# Práctica 05 — Clasificación de textos

**Curso:** NLP Clásico con Python · **Clase 05** · **Tiempo estimado:** 75-90 minutos
**Notebook de clase:** `notebooks_clase/clase05_clasificacion_de_textos.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- El ejercicio 1 usa un corpus minúsculo definido en la celda de datos; los demás usan los tickets de `datos/` (en Colab, sube la carpeta `datos/`). Son textos sintéticos escritos para el curso (*ejemplo ilustrativo*), con español e italiano casi paralelos: no son dos experimentos independientes.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas` y `scikit-learn`.

---
## Celda 0 — Setup y datos

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

ARCHIVO_TICKETS = {"spa": "tickets_es.csv", "ita": "tickets_it.csv"}
tickets = {idioma: pd.read_csv(cargar_datos(archivo)) for idioma, archivo in ARCHIVO_TICKETS.items()}
ETIQUETAS = sorted(tickets["spa"]["categoria"].unique())


def separar_veredicto(texto):
    """Separa la última frase si tiene 4 palabras o menos (veredicto o etiqueta de asunto)."""
    m = re.search(r"(?<=[.?!])\s+([^.?!]+[.?!])\s*$", texto)
    if m and len(m.group(1).split()) <= 4:
        return texto[:m.start()].strip(), m.group(1).strip()
    return texto, ""


print({idioma: df.shape for idioma, df in tickets.items()})
print("Librerías cargadas correctamente.")

In [ ]:
# datos-es
NB_P = {
    "spa": {"cuenta": ["olvidé mi contraseña", "no puedo entrar en mi cuenta"],
            "envio": ["mi paquete no llega", "el paquete llega tarde"]},
    "ita": {"cuenta": ["ho dimenticato la password", "non riesco ad entrare nel mio account"],
            "envio": ["il mio pacco non arriva", "il pacco arriva in ritardo"]},
}
TEXTO_NB = {"spa": "no llega mi cuenta", "ita": "non arriva il mio account"}
CUOTAS_P = {"facturacion": 80, "envio": 40, "cuenta": 15, "devoluciones": 8}

## Ejercicio 1 — Naive Bayes con suavizado de Laplace, a mano

Con el corpus minúsculo `NB_P` (dos tickets por clase, `cuenta` y `envio`) y el ticket nuevo de `TEXTO_NB`:

- completa `prob_palabra(recuento, n_clase, V, alpha)`: `(recuento + alpha) / (n_clase + alpha · V)`,
- completa el **priori** de `puntuacion` (fracción de documentos de la clase).

`puntuacion` multiplica el priori por la probabilidad de cada palabra del ticket. Se comprueba con `MultinomialNB(alpha=1)` en los dos idiomas. Antes de programar, calcula a mano `P(no | cuenta)` en español: la clase `cuenta` tiene 9 palabras y el vocabulario 12.

In [ ]:
def prob_palabra(recuento, n_clase, V, alpha=1.0):
    return None      # TODO: (recuento + alpha) / (n_clase + alpha · V)


def preparar_nb(idioma):
    docs = {clase: [re.findall(r"\w+", d.lower()) for d in ds] for clase, ds in NB_P[idioma].items()}
    vocab = sorted({t for ds in docs.values() for d in ds for t in d})
    return docs, vocab


def puntuacion(clase, texto, docs, V, alpha=1.0):
    tokens_clase = [t for d in docs[clase] for t in d]
    priori = None    # TODO: documentos de la clase / documentos totales
    p = priori
    for palabra in re.findall(r"\w+", texto.lower()):
        p = p * prob_palabra(tokens_clase.count(palabra), len(tokens_clase), V, alpha)
    return p

# Comprobación automática
assert prob_palabra(2, 6, 11) is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(prob_palabra(2, 6, 11), 3 / 17), "(2 + 1) / (6 + 11) = 3/17"
assert np.isclose(prob_palabra(0, 6, 11), 1 / 17) and prob_palabra(0, 6, 11, alpha=0) == 0, "con alpha = 0 no hay suavizado y la probabilidad es 0"
for idioma in ("spa", "ita"):
    docs, vocab = preparar_nb(idioma)
    V = len(vocab)
    texto = TEXTO_NB[idioma]
    assert puntuacion("cuenta", texto, docs, V) is not None and puntuacion("envio", texto, docs, V) is not None, "Completa las líneas marcadas con TODO"
    puntos = {c: puntuacion(c, texto, docs, V) for c in sorted(docs)}
    total = sum(puntos.values())
    posteriores = np.array([puntos[c] / total for c in sorted(docs)])
    textos = [d for ds in NB_P[idioma].values() for d in ds]
    etiquetas = [c for c, ds in NB_P[idioma].items() for _ in ds]
    cv = CountVectorizer(token_pattern=r"\w+")
    nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(textos), etiquetas)
    assert list(nb.classes_) == sorted(docs)
    assert np.allclose(posteriores, nb.predict_proba(cv.transform([texto]))[0]), "los posteriores deben coincidir con MultinomialNB(alpha=1): revisa el priori y la fórmula"
    assert np.isclose(puntos["cuenta"] / total + puntos["envio"] / total, 1.0)
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Precisión, recall y F1 desde la matriz de confusión

`M` es una matriz de confusión con **clases desbalanceadas** (filas = clase real, columnas = clase predicha, en el orden de `ETIQUETAS`). Calcula, sin usar `scikit-learn`:

- `precision`, `recall` y `f1`: arrays con un valor por clase,
- `f1_macro`: media simple de los F1,
- `f1_pond`: media de los F1 ponderada por el soporte (número de ejemplos reales de cada clase).

Antes de programar, calcula a mano el recall de `devoluciones` (fila 2).

In [ ]:
M = np.array([[24, 3, 2, 1],      # cuenta
              [1, 6, 1, 2],       # devoluciones
              [2, 0, 17, 1],      # envio
              [1, 2, 0, 37]])     # facturacion
print(ETIQUETAS)

precision = None    # TODO: diagonal / suma de cada columna
recall = None       # TODO: diagonal / suma de cada fila
f1 = None           # TODO: 2 · P · R / (P + R)
f1_macro = None     # TODO
f1_pond = None      # TODO: ponderado por el soporte (suma de cada fila)

# Comprobación automática
assert all(v is not None for v in (precision, recall, f1, f1_macro, f1_pond)), "Completa las líneas marcadas con TODO"
assert np.isclose(recall[1], 6 / 10), "el recall de devoluciones es 6/10 = 0,6"
y_real = [ETIQUETAS[i] for i in range(4) for j in range(4) for _ in range(M[i, j])]
y_pred = [ETIQUETAS[j] for i in range(4) for j in range(4) for _ in range(M[i, j])]
p, r, f, s = precision_recall_fscore_support(y_real, y_pred, labels=ETIQUETAS)
assert np.allclose(precision, p), "precisión = diagonal / suma de la columna"
assert np.allclose(recall, r), "recall = diagonal / suma de la fila"
assert np.allclose(f1, f), "F1 = 2 · P · R / (P + R)"
assert np.isclose(f1_macro, f1_score(y_real, y_pred, average="macro")), "el F1 macro es la media simple de los F1"
assert np.isclose(f1_pond, f1_score(y_real, y_pred, average="weighted")), "el ponderado usa como pesos el soporte de cada clase"
assert f1_macro < f1_pond, "aquí el ponderado es mayor: la clase pequeña (devoluciones) es la peor"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — `Pipeline` y validación cruzada estratificada

Completa:

- `crear_pipeline_svm()`: una `Pipeline` con dos pasos, `"tfidf"` (`TfidfVectorizer(ngram_range=(1, 2))`) y `"clf"` (`LinearSVC()`),
- `validar(df)`: devuelve las 5 exactitudes de `cross_val_score` con `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, pasando **la `Pipeline` entera** (así el vectorizador se ajusta solo con el entrenamiento de cada partición).

La comprobación repite el cálculo a mano, ajustando en cada partición solo con el entrenamiento, y debe dar lo mismo.

In [ ]:
def crear_pipeline_svm():
    return None      # TODO: Pipeline([("tfidf", ...), ("clf", ...)])


def validar(df):
    return None      # TODO: cross_val_score(..., cv=StratifiedKFold(...))

# Comprobación automática
pipe = crear_pipeline_svm()
assert pipe is not None, "Completa las líneas marcadas con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline debe tener los pasos «tfidf» y «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2)
assert isinstance(pipe.named_steps["clf"], LinearSVC)
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline se devuelve sin ajustar"
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    puntos = validar(df)
    assert puntos is not None, "Completa las líneas marcadas con TODO"
    assert len(puntos) == 5, "5 particiones, 5 valores"
    manual = []
    for idx_ent, idx_prueba in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(df["texto"], df["categoria"]):
        modelo = crear_pipeline_svm().fit(df["texto"].iloc[idx_ent], df["categoria"].iloc[idx_ent])
        manual.append(modelo.score(df["texto"].iloc[idx_prueba], df["categoria"].iloc[idx_prueba]))
    assert np.allclose(puntos, manual), "debe coincidir con el cálculo manual: pasa la Pipeline entera a cross_val_score"
    assert puntos.mean() > 0.8, "con TF-IDF y LinearSVC se espera más de 0,8"
    print(idioma, "exactitud media", round(puntos.mean(), 3), "±", round(puntos.std(), 3))
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Crear un desbalance y medir lo que la exactitud esconde

Completa:

- `submuestrear(df, cuotas, semilla)`: se queda con `cuotas[categoria]` tickets de cada categoría (usa `groupby` y `sample(n=..., random_state=semilla)`) y devuelve un `DataFrame` con el índice reiniciado,
- `evaluar(des)`: entrena una regresión logística en una `Pipeline` (TF-IDF con unigramas y bigramas) **sin** pesos y otra con `class_weight="balanced"`, y devuelve `(exactitud_sin, f1_macro_sin, f1_macro_balanced)` sobre la prueba.

Antes de programar: ¿qué exactitud tendría un modelo que dice siempre `facturacion` con `CUOTAS_P`? (80 de 143 tickets.)

In [ ]:
def submuestrear(df, cuotas, semilla=42):
    return None      # TODO: concatenar g.sample(n=cuotas[c], ...) de cada categoría


def evaluar(des):
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(des["texto"], des["categoria"], test_size=0.3, stratify=des["categoria"], random_state=42)
    exactitud_sin = None        # TODO: exactitud de la regresión logística sin pesos
    f1_sin = None               # TODO: F1 macro sin pesos
    f1_balanced = None          # TODO: F1 macro con class_weight="balanced"
    return exactitud_sin, f1_sin, f1_balanced

# Comprobación automática
for idioma in ("spa", "ita"):
    des = submuestrear(tickets[idioma], CUOTAS_P)
    assert des is not None, "Completa las líneas marcadas con TODO"
    assert des["categoria"].value_counts().to_dict() == CUOTAS_P, "el DataFrame debe tener exactamente las cuotas pedidas"
    assert list(des.index) == list(range(143)), "reinicia el índice (reset_index(drop=True))"
    assert des.equals(submuestrear(tickets[idioma], CUOTAS_P)), "con la misma semilla el resultado debe repetirse"
    res = evaluar(des)
    assert all(v is not None for v in res), "Completa las líneas marcadas con TODO"
    exactitud_sin, f1_sin, f1_bal = res
    assert exactitud_sin > f1_sin + 0.15, "sin pesos la exactitud debería ser bastante mayor que el F1 macro"
    assert f1_bal > f1_sin + 0.1, "class_weight=\"balanced\" debería mejorar el F1 macro"
    print(idioma, "exactitud sin pesos", round(exactitud_sin, 3), "| F1 macro sin pesos", round(f1_sin, 3), "| F1 macro balanced", round(f1_bal, 3))
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Leer coeficientes y detectar artefactos

Completa:

- `top_palabras(pipe, k)`: dada una `Pipeline` ya ajustada (TF-IDF + `LogisticRegression`), devuelve un diccionario `clase → lista de las k palabras con mayor coeficiente` (de mayor a menor),
- `n_veredictos(df)`: número de tickets cuyo texto tiene una frase final corta según `separar_veredicto` (el segundo elemento que devuelve no está vacío).

Es la lectura crítica de la clase: las palabras de la lista se interpretan, y los veredictos y las etiquetas de asunto son un artefacto de los datos.

In [ ]:
def top_palabras(pipe, k=8):
    palabras = pipe.named_steps["tfidf"].get_feature_names_out()
    coef = pipe.named_steps["clf"].coef_
    clases = pipe.named_steps["clf"].classes_
    return None      # TODO: {clase: [k palabras con mayor coeficiente]}


def n_veredictos(df):
    return None      # TODO: cuenta los tickets con veredicto

# Comprobación automática
FACTURACION = {"spa": {"cobro", "factura", "pago", "importe", "tarjeta", "cargo", "cobrado", "pagos"},
               "ita": {"addebito", "fattura", "pagamento", "importo", "addebitato", "carta", "pagato", "pagare"}}
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]).fit(df["texto"], df["categoria"])
    top = top_palabras(pipe, k=8)
    assert top is not None and n_veredictos(df) is not None, "Completa las líneas marcadas con TODO"
    assert sorted(top) == ETIQUETAS and all(len(v) == 8 for v in top.values()), "una lista de 8 palabras por clase"
    fila = list(pipe.named_steps["clf"].classes_).index("facturacion")
    coef = pipe.named_steps["clf"].coef_[fila]
    palabras = list(pipe.named_steps["tfidf"].get_feature_names_out())
    assert top["facturacion"][0] == palabras[int(np.argmax(coef))], "la primera palabra debe ser la de mayor coeficiente"
    coefs_top = [coef[palabras.index(p)] for p in top["facturacion"]]
    assert coefs_top == sorted(coefs_top, reverse=True), "ordena de mayor a menor coeficiente"
    assert len(set(top["facturacion"][:5]) & FACTURACION[idioma]) >= 3, "las palabras de facturación deberían dominar la lista"
    assert n_veredictos(df) == sum(1 for t in df["texto"] if separar_veredicto(t)[1]) and n_veredictos(df) > 100, "cuenta los tickets con frase final corta"
    print(idioma, "facturacion:", top["facturacion"], "| tickets con veredicto:", n_veredictos(df))
print("Ejercicio 5 correcto ✔")

---
## Reto final — ¿cuánto vale el veredicto?

Integra lo aprendido. Completa `diferencia_veredicto(df)`, que debe devolver `(con, sin)`:

- `cuerpos`: la serie de textos **sin** la frase final corta (usa `separar_veredicto`),
- `con`: exactitud media de validación cruzada estratificada (5 particiones, `shuffle=True`, `random_state=42`) con una `Pipeline` TF-IDF + regresión logística sobre los textos completos,
- `sin`: la misma medida sobre `cuerpos`.

Comprueba que en los dos idiomas quitar el veredicto **baja** la exactitud. Después, escribe en la celda de conclusiones qué significa esto para interpretar los coeficientes.

In [ ]:
def diferencia_veredicto(df):
    cuerpos = None      # TODO: textos sin la frase final corta
    con = None          # TODO: exactitud media (CV estratificada) con los textos completos
    sin = None          # TODO: exactitud media (CV estratificada) con «cuerpos»
    return con, sin

def _cv(textos, y):
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000))])
    return cross_val_score(pipe, textos, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)).mean()


for idioma in ("spa", "ita"):
    df = tickets[idioma]
    resultado = diferencia_veredicto(df)
    assert resultado is not None, "Completa las líneas marcadas con TODO"
    con, sin = resultado
    assert con is not None and sin is not None, "Completa las líneas marcadas con TODO"
    esperado_sin = _cv(df["texto"].map(lambda t: separar_veredicto(t)[0]), df["categoria"])
    assert np.isclose(con, _cv(df["texto"], df["categoria"])), "valor con veredicto: validación cruzada sobre los textos completos"
    assert np.isclose(sin, esperado_sin), "valor sin veredicto: validación cruzada sobre los textos sin la frase final corta"
    assert con - sin > 0.01, "quitar el veredicto debería bajar la exactitud"
    print(idioma, "con veredicto", round(con, 4), "| sin veredicto", round(sin, 4))
print("Reto final correcto ✔")

**Tus conclusiones (escribe 3-4 frases):** ¿qué significa que la exactitud baje al quitar la frase final? Si un coeficiente alto de `cobro` o `acceso` viene en parte de esa frase, ¿lo usarías para explicar qué hace el modelo con tickets reales? ¿Qué otros artefactos buscarías en un corpus tuyo antes de interpretar coeficientes?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def prob_palabra(recuento, n_clase, V, alpha=1.0):
    return (recuento + alpha) / (n_clase + alpha * V)


def preparar_nb(idioma):
    docs = {clase: [re.findall(r"\w+", d.lower()) for d in ds] for clase, ds in NB_P[idioma].items()}
    vocab = sorted({t for ds in docs.values() for d in ds for t in d})
    return docs, vocab


def puntuacion(clase, texto, docs, V, alpha=1.0):
    tokens_clase = [t for d in docs[clase] for t in d]
    priori = len(docs[clase]) / sum(len(ds) for ds in docs.values())
    p = priori
    for palabra in re.findall(r"\w+", texto.lower()):
        p = p * prob_palabra(tokens_clase.count(palabra), len(tokens_clase), V, alpha)
    return p

# Comprobación automática
assert prob_palabra(2, 6, 11) is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(prob_palabra(2, 6, 11), 3 / 17), "(2 + 1) / (6 + 11) = 3/17"
assert np.isclose(prob_palabra(0, 6, 11), 1 / 17) and prob_palabra(0, 6, 11, alpha=0) == 0, "con alpha = 0 no hay suavizado y la probabilidad es 0"
for idioma in ("spa", "ita"):
    docs, vocab = preparar_nb(idioma)
    V = len(vocab)
    texto = TEXTO_NB[idioma]
    assert puntuacion("cuenta", texto, docs, V) is not None and puntuacion("envio", texto, docs, V) is not None, "Completa las líneas marcadas con TODO"
    puntos = {c: puntuacion(c, texto, docs, V) for c in sorted(docs)}
    total = sum(puntos.values())
    posteriores = np.array([puntos[c] / total for c in sorted(docs)])
    textos = [d for ds in NB_P[idioma].values() for d in ds]
    etiquetas = [c for c, ds in NB_P[idioma].items() for _ in ds]
    cv = CountVectorizer(token_pattern=r"\w+")
    nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(textos), etiquetas)
    assert list(nb.classes_) == sorted(docs)
    assert np.allclose(posteriores, nb.predict_proba(cv.transform([texto]))[0]), "los posteriores deben coincidir con MultinomialNB(alpha=1): revisa el priori y la fórmula"
    assert np.isclose(puntos["cuenta"] / total + puntos["envio"] / total, 1.0)
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
M = np.array([[24, 3, 2, 1],      # cuenta
              [1, 6, 1, 2],       # devoluciones
              [2, 0, 17, 1],      # envio
              [1, 2, 0, 37]])     # facturacion
print(ETIQUETAS)

soporte = M.sum(axis=1)
precision = np.diag(M) / M.sum(axis=0)
recall = np.diag(M) / soporte
f1 = 2 * precision * recall / (precision + recall)
f1_macro = f1.mean()
f1_pond = (f1 * soporte).sum() / soporte.sum()

# Comprobación automática
assert all(v is not None for v in (precision, recall, f1, f1_macro, f1_pond)), "Completa las líneas marcadas con TODO"
assert np.isclose(recall[1], 6 / 10), "el recall de devoluciones es 6/10 = 0,6"
y_real = [ETIQUETAS[i] for i in range(4) for j in range(4) for _ in range(M[i, j])]
y_pred = [ETIQUETAS[j] for i in range(4) for j in range(4) for _ in range(M[i, j])]
p, r, f, s = precision_recall_fscore_support(y_real, y_pred, labels=ETIQUETAS)
assert np.allclose(precision, p), "precisión = diagonal / suma de la columna"
assert np.allclose(recall, r), "recall = diagonal / suma de la fila"
assert np.allclose(f1, f), "F1 = 2 · P · R / (P + R)"
assert np.isclose(f1_macro, f1_score(y_real, y_pred, average="macro")), "el F1 macro es la media simple de los F1"
assert np.isclose(f1_pond, f1_score(y_real, y_pred, average="weighted")), "el ponderado usa como pesos el soporte de cada clase"
assert f1_macro < f1_pond, "aquí el ponderado es mayor: la clase pequeña (devoluciones) es la peor"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def crear_pipeline_svm():
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LinearSVC())])


def validar(df):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(crear_pipeline_svm(), df["texto"], df["categoria"], cv=skf)

# Comprobación automática
pipe = crear_pipeline_svm()
assert pipe is not None, "Completa las líneas marcadas con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline debe tener los pasos «tfidf» y «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2)
assert isinstance(pipe.named_steps["clf"], LinearSVC)
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline se devuelve sin ajustar"
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    puntos = validar(df)
    assert puntos is not None, "Completa las líneas marcadas con TODO"
    assert len(puntos) == 5, "5 particiones, 5 valores"
    manual = []
    for idx_ent, idx_prueba in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(df["texto"], df["categoria"]):
        modelo = crear_pipeline_svm().fit(df["texto"].iloc[idx_ent], df["categoria"].iloc[idx_ent])
        manual.append(modelo.score(df["texto"].iloc[idx_prueba], df["categoria"].iloc[idx_prueba]))
    assert np.allclose(puntos, manual), "debe coincidir con el cálculo manual: pasa la Pipeline entera a cross_val_score"
    assert puntos.mean() > 0.8, "con TF-IDF y LinearSVC se espera más de 0,8"
    print(idioma, "exactitud media", round(puntos.mean(), 3), "±", round(puntos.std(), 3))
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def submuestrear(df, cuotas, semilla=42):
    partes = [g.sample(n=cuotas[c], random_state=semilla) for c, g in df.groupby("categoria")]
    return pd.concat(partes).reset_index(drop=True)


def evaluar(des):
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(des["texto"], des["categoria"], test_size=0.3, stratify=des["categoria"], random_state=42)
    resultados = []
    for peso in (None, "balanced"):
        pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000, class_weight=peso))]).fit(X_ent, y_ent)
        resultados.append(pipe.predict(X_prueba))
    exactitud_sin = accuracy_score(y_prueba, resultados[0])
    f1_sin = f1_score(y_prueba, resultados[0], average="macro")
    f1_balanced = f1_score(y_prueba, resultados[1], average="macro")
    return exactitud_sin, f1_sin, f1_balanced

# Comprobación automática
for idioma in ("spa", "ita"):
    des = submuestrear(tickets[idioma], CUOTAS_P)
    assert des is not None, "Completa las líneas marcadas con TODO"
    assert des["categoria"].value_counts().to_dict() == CUOTAS_P, "el DataFrame debe tener exactamente las cuotas pedidas"
    assert list(des.index) == list(range(143)), "reinicia el índice (reset_index(drop=True))"
    assert des.equals(submuestrear(tickets[idioma], CUOTAS_P)), "con la misma semilla el resultado debe repetirse"
    res = evaluar(des)
    assert all(v is not None for v in res), "Completa las líneas marcadas con TODO"
    exactitud_sin, f1_sin, f1_bal = res
    assert exactitud_sin > f1_sin + 0.15, "sin pesos la exactitud debería ser bastante mayor que el F1 macro"
    assert f1_bal > f1_sin + 0.1, "class_weight=\"balanced\" debería mejorar el F1 macro"
    print(idioma, "exactitud sin pesos", round(exactitud_sin, 3), "| F1 macro sin pesos", round(f1_sin, 3), "| F1 macro balanced", round(f1_bal, 3))
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def top_palabras(pipe, k=8):
    palabras = pipe.named_steps["tfidf"].get_feature_names_out()
    coef = pipe.named_steps["clf"].coef_
    clases = pipe.named_steps["clf"].classes_
    return {clase: [palabras[j] for j in np.argsort(coef[i])[::-1][:k]] for i, clase in enumerate(clases)}


def n_veredictos(df):
    return sum(1 for t in df["texto"] if separar_veredicto(t)[1] != "")

# Comprobación automática
FACTURACION = {"spa": {"cobro", "factura", "pago", "importe", "tarjeta", "cargo", "cobrado", "pagos"},
               "ita": {"addebito", "fattura", "pagamento", "importo", "addebitato", "carta", "pagato", "pagare"}}
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]).fit(df["texto"], df["categoria"])
    top = top_palabras(pipe, k=8)
    assert top is not None and n_veredictos(df) is not None, "Completa las líneas marcadas con TODO"
    assert sorted(top) == ETIQUETAS and all(len(v) == 8 for v in top.values()), "una lista de 8 palabras por clase"
    fila = list(pipe.named_steps["clf"].classes_).index("facturacion")
    coef = pipe.named_steps["clf"].coef_[fila]
    palabras = list(pipe.named_steps["tfidf"].get_feature_names_out())
    assert top["facturacion"][0] == palabras[int(np.argmax(coef))], "la primera palabra debe ser la de mayor coeficiente"
    coefs_top = [coef[palabras.index(p)] for p in top["facturacion"]]
    assert coefs_top == sorted(coefs_top, reverse=True), "ordena de mayor a menor coeficiente"
    assert len(set(top["facturacion"][:5]) & FACTURACION[idioma]) >= 3, "las palabras de facturación deberían dominar la lista"
    assert n_veredictos(df) == sum(1 for t in df["texto"] if separar_veredicto(t)[1]) and n_veredictos(df) > 100, "cuenta los tickets con frase final corta"
    print(idioma, "facturacion:", top["facturacion"], "| tickets con veredicto:", n_veredictos(df))
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def diferencia_veredicto(df):
    cuerpos = df["texto"].map(lambda t: separar_veredicto(t)[0])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000))])
    con = cross_val_score(pipe, df["texto"], df["categoria"], cv=skf).mean()
    sin = cross_val_score(pipe, cuerpos, df["categoria"], cv=skf).mean()
    return con, sin

def _cv(textos, y):
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000))])
    return cross_val_score(pipe, textos, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)).mean()


for idioma in ("spa", "ita"):
    df = tickets[idioma]
    resultado = diferencia_veredicto(df)
    assert resultado is not None, "Completa las líneas marcadas con TODO"
    con, sin = resultado
    assert con is not None and sin is not None, "Completa las líneas marcadas con TODO"
    esperado_sin = _cv(df["texto"].map(lambda t: separar_veredicto(t)[0]), df["categoria"])
    assert np.isclose(con, _cv(df["texto"], df["categoria"])), "valor con veredicto: validación cruzada sobre los textos completos"
    assert np.isclose(sin, esperado_sin), "valor sin veredicto: validación cruzada sobre los textos sin la frase final corta"
    assert con - sin > 0.01, "quitar el veredicto debería bajar la exactitud"
    print(idioma, "con veredicto", round(con, 4), "| sin veredicto", round(sin, 4))
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (Naive Bayes y suavizado, precisión frente a recall, F1 macro frente a ponderado, fuga de datos, coeficientes, validación cruzada)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*